# ⭐PostgreSQL이 임베딩 DB로 좋은 이유 & pgvector 기초 (시연)

> **어제까지 표(PostgreSQL)·key-value(Redis)로 데이터를 다뤘습니다. 오늘은 AI 전용 데이터 — '의미의 좌표(임베딩)'를 만납니다.** 텍스트를 숫자 벡터로 바꿔 저장하고, 질문도 벡터로 바꿔 **의미가 가장 가까운 문서**를 찾습니다(키워드가 하나도 안 겹쳐도!). 그리고 그 벡터를 **왜 별도 벡터 DB가 아니라 이미 쓰는 PostgreSQL에 두면 좋은지**를 6가지 근거로 보고, `CREATE EXTENSION vector` 한 줄로 PostgreSQL을 벡터 DB로 만듭니다.

## 차시 학습 목표
- ⭐**PostgreSQL이 임베딩 DB로 좋은 이유 6가지**를 이해한다
- `db-pg`에 접속해 `CREATE EXTENSION vector`로 벡터 확장을 켜고, `vector(1024)` 타입·연산자 4종을 다룬다
- BGE-M3로 도서관 6문서를 임베딩·적재하고, `<=>` 코사인으로 **미니 시맨틱 검색**을 한다
- ⭐`WHERE`(메타 필터) + 벡터 검색을 **하나의 SQL로 결합**하고, HNSW 인덱스를 만든다

## 다루는 내용
- ⭐PostgreSQL이 임베딩 DB로 좋은 6가지 이유 (이론)
- pgvector 시작 — 확장·벡터 타입·연산자
- 미니 시맨틱 검색 — 문서 임베딩→적재→검색
- 메타데이터 필터 + 벡터 결합 & HNSW 인덱스 (⭐6축 #5 실증)

> ⚠️ **표기 규약**: **검색 순위(어느 문서가 1등인가)는 확정값**입니다. 하지만 **유사도 소수 수치는 '근사(약 0.70·±0.01)'** 로 봅니다 — 부동소수점·모델 버전·하드웨어에 따라 뒷자리가 미세하게 달라질 수 있으므로 **'수치가 아니라 순위'** 에 집중하세요. 모델 로딩·인덱스 빌드 **시간은 '예시'**(기기마다 다름)입니다.
>
> 📌 이 Day는 **임베딩(BGE-M3)만** 사용하고 **LLM은 쓰지 않습니다** — 오늘 범위는 '검색까지'(retrieval)이며, 찾은 문서로 답을 생성하는 것은 rag-course 소관입니다.

## 환경 준비 — 패키지 설치

이 시연에 필요한 3개 패키지입니다.

In [1]:
# ✅ 포인트: 임베딩(sentence-transformers)·벡터 바인딩(pgvector)·DB 드라이버(psycopg) 3종.
#   - sentence-transformers : 문장을 숫자 벡터(임베딩)로 바꿔주는 BGE-M3 모델을 불러올 때 사용
#   - pgvector              : PostgreSQL의 vector 타입을 파이썬 numpy 배열과 주고받게 해주는 어댑터
#   - "psycopg[binary]"     : 파이썬에서 PostgreSQL에 접속·SQL 실행을 담당하는 드라이버
#   - python-dotenv         : .env 파일에 적어둔 비밀번호를 안전하게 읽어오는 도구
%pip install -q sentence-transformers pgvector "psycopg[binary]" python-dotenv

Note: you may need to restart the kernel to use updated packages.


## ⭐PostgreSQL이 임베딩 DB로 좋은 6가지 이유 (이론)

임베딩(벡터)을 **어디에** 저장할까요? 새 벡터 DB를 또 세울까요, 아니면 **이미 쓰고 있는 PostgreSQL에 얹을까요?** PostgreSQL은 `CREATE EXTENSION vector` 한 줄이면 벡터 DB가 됩니다. 왜 그게 좋은지 6가지 근거입니다.

| # | 근거 | 한 줄 설명 |
|---|------|-----------|
| 1 | **정형 데이터와 한 곳에 산다** | 임베딩이 업무 데이터와 같은 DB에 있어 **JOIN**·**외래 키**로 정합성 유지. 별도 벡터 DB로의 **이중 쓰기(dual-write)·동기화 문제가 없다**. |
| 2 | **통합 인프라** | 트랜잭션 데이터와 AI 워크로드를 한 시스템으로 — **백업·보안·모니터링을 단순화**(운영 복잡도·비용↓). 새 DB 도입 불필요. |
| 3 | **ACID 트랜잭션** | 원자성·일관성·격리성·지속성 보장. 전용 벡터 DB가 항상 주지는 않는 데이터 무결성. |
| 4 | **기존 SQL 스킬·도구 재사용** | 익숙한 SQL·드라이버·백업 도구를 그대로. 이미 PostgreSQL 쓰는 팀엔 **확장 설치만으로** 도입. |
| 5 | **메타데이터 필터 + 벡터 검색 결합** | 하나의 SQL로 `WHERE`(필터)와 벡터 유사도(`<=>`)를 함께. **(뒤의 메타데이터 필터 + 벡터 결합에서 코드로 실증)** |
| 6 | **적정 규모 성능** | 수백만 임베딩까지 HNSW/IVFFlat로 충분. (**한계 균형**: 초대규모·초고QPS는 전용 벡터 DB가 유리) |

> **✅ 포인트(⭐)**: 특히 **#1(한 곳에 산다)·#5(필터+벡터 결합)** 이 실무 최대 이점입니다. 업무 테이블과 임베딩을 JOIN하고, '이 카테고리에서 의미가 가까운 것'을 **한 SQL로** 뽑을 수 있죠. 균형도 잊지 마세요 — 초대규모(수억+)·초고QPS는 전용 벡터 DB(Pinecone 등)가 유리합니다. 도구는 규모에 맞게!

## pgvector 시작 — 확장·벡터 타입·연산자

이제 코드로 들어갑니다. Docker로 PostgreSQL 띄우기에서 세운 그 **`db-pg`의 `library` DB**에 접속해서, `CREATE EXTENSION vector`로 벡터 확장을 켜고, `documents` 테이블을 만듭니다.

> 📌 오늘은 **PostgreSQL로 복귀**했으니 비밀번호가 필요합니다 — `.env` 파일에서 읽습니다(Docker로 PostgreSQL 띄우기 규약 부활). 어제 Redis는 무비밀번호였던 것과 헷갈리지 마세요!

In [2]:
# ✅ 포인트: 비밀번호를 코드에 쓰지 않습니다 — 같은 폴더의 .env 파일에 두고 load_dotenv()로 읽습니다.
# 💡 .env는 이 폴더의 텍스트 파일(PGPASSWORD=... 형태)입니다 — Git에 올리지 않아요.
import os
from dotenv import load_dotenv   # .env 파일 내용을 환경변수로 불러오는 함수
import psycopg                    # PostgreSQL 접속·SQL 실행 드라이버

load_dotenv()                                   # 같은 폴더의 .env → 환경변수
pw = os.environ["PGPASSWORD"]                   # 비밀번호는 .env 파일에 (하드코딩 금지)

# ✅ 포인트: Docker로 PostgreSQL 띄우기에서 세운 db-pg의 library DB에 접속(기존 books/members/rentals는 건드리지 않습니다).
# host(주소)·port(포트)·dbname(DB 이름)을 지정해 접속을 시도하고, 성공하면 conn(연결) 객체를 돌려받습니다.
conn = psycopg.connect(host="localhost", port=5432, dbname="library", user="postgres", password=pw)
print("연결 완료 →", conn)

연결 완료 → <psycopg.Connection [IDLE] (host=localhost user=postgres database=library) at 0x195837a1400>


In [3]:
# ✅ 포인트: PostgreSQL이 '벡터 DB'가 되는 한 줄 — CREATE EXTENSION vector (확장 0.8.5 내장·멱등).
from pgvector.psycopg import register_vector   # numpy 배열 ↔ PostgreSQL vector 타입을 자동 변환해주는 등록 함수

conn.execute("CREATE EXTENSION IF NOT EXISTS vector")   # 벡터 확장 활성화(library DB에 한 번)
conn.commit()                                            # DDL도 커밋(autocommit=False가 기본)

# ✅ 포인트: register_vector는 '확장 활성화 후'에 호출 — numpy 배열을 vector 파라미터로 바인딩해줍니다.
register_vector(conn)

# 확장 버전 확인 → 0.8.5 (이미지에 내장·이미지 교체나 재-pull 불필요)
# pg_extension은 설치된 확장 목록을 담은 시스템 테이블. WHERE extname='vector'로 그중 한 줄만 골라 버전 확인.
ver = conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0]
print("pgvector 확장 버전 →", ver)

pgvector 확장 버전 → 0.8.6


In [4]:
# ✅ 포인트: documents 테이블 — embedding vector(1024)가 핵심(BGE-M3가 1024차원 벡터를 냅니다).
# 💡 멱등 준비: DROP TABLE IF EXISTS로 재실행해도 안전. 기존 books/members/rentals와 무관한 새 테이블입니다.
conn.execute("DROP TABLE IF EXISTS documents")
conn.execute("""
CREATE TABLE documents (
    id        BIGSERIAL PRIMARY KEY,   -- 문서 번호(자동 증가)
    content   TEXT,                    -- 문서 원문
    embedding vector(1024),            -- ✅ 의미의 좌표(BGE-M3 1024차원)
    category  TEXT                     -- 메타데이터(필터용 카테고리)
)
""")
conn.commit()
print("documents 테이블 생성 완료")

documents 테이블 생성 완료


### 연산자 4종 — 오늘은 코사인 `<=>` 를 씁니다

pgvector는 벡터 사이의 '거리'를 재는 연산자를 4개 제공합니다. **작을수록 가까움**을 뜻합니다.

| 연산자 | 의미 | 비고 |
|--------|------|------|
| **`<=>`** | **코사인 거리** | ✅ 오늘 사용(BGE-M3 권장 지표). `1 - (a <=> b)` = 코사인 유사도 |
| `<->` | L2(유클리드) 거리 | 이름만 |
| `<#>` | 음의 내적 | 이름만 |
| `<+>` | L1(맨해튼) 거리 | 이름만 |

> **✅ 포인트**: `CREATE EXTENSION vector` → `vector(1024)` 타입 → `<=>` 코사인. PostgreSQL에 **한 줄**이면 벡터 DB가 되고, Docker로 PostgreSQL 띄우기에서 세운 바로 그 `db-pg`에 얹습니다.

## 미니 시맨틱 검색 — 문서 임베딩→적재→검색

이제 진짜 시맨틱 검색입니다. 도서관 6문서를 BGE-M3로 임베딩해 `documents`에 적재하고, 질문도 임베딩해서 `<=>` 코사인으로 **의미가 가장 가까운 문서**를 찾습니다.

> 📌 **오프라인 env를 모델 로딩보다 먼저!** `HF_HUB_OFFLINE`·`TRANSFORMERS_OFFLINE`을 `SentenceTransformer` import/로딩 **전에** 설정해야 합니다(미설정 시 허브 접속을 시도해 100초+ 지연 — rag-course 실증).

In [5]:
# 📌 오프라인 env를 모델 로딩보다 먼저! (미설정 시 100초+ 지연)
import os
os.environ["HF_HUB_OFFLINE"] = "1"        # HuggingFace 허브 오프라인
os.environ["TRANSFORMERS_OFFLINE"] = "1"  # transformers 오프라인

# ✅ 포인트: env 설정 '후에' import·로딩 — 로컬 캐시의 BGE-M3를 씁니다.
from sentence_transformers import SentenceTransformer   # 문장 → 숫자 벡터로 바꿔주는 임베딩 모델 클래스

MODEL = "BAAI/bge-m3"                      # 임베딩 모델(다국어·1024차원)
model = SentenceTransformer(MODEL)         # 📌 로딩 약 5~10초(예시·기기마다 다름)
print("모델 로딩 완료:", MODEL)

# 차원 확인 → 1024 (vector(1024) 스키마와 정합)
# model.encode(문장)은 벡터를 돌려주고, .shape[0]으로 그 벡터가 숫자 몇 개로 이루어졌는지 확인합니다.
dim = model.encode("차원 확인용 문장").shape[0]
print("임베딩 차원 →", dim)

c:\Users\Admin\Desktop\실습용\my_llm_service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 29170.11it/s]


모델 로딩 완료: BAAI/bge-m3
임베딩 차원 → 1024


In [6]:
# ✅ 포인트: 도서관 안내 6문서
# 각 문서는 (내용, 카테고리) 쌍입니다. 곧이어 이 6개를 벡터로 바꿔 documents 테이블에 넣고,
# 질문과 의미가 가장 비슷한 문서를 찾는 시맨틱 검색을 연습합니다.
docs = [
    ("도서관 운영 시간은 평일 오전 9시부터 오후 10시까지입니다.", "이용안내"),
    ("회원 가입은 신분증을 지참하면 1층 안내데스크에서 즉시 가능합니다.", "이용안내"),
    ("한 번에 최대 5권까지 2주 동안 대출할 수 있습니다.", "대출"),
    ("노트북을 사용할 수 있는 디지털 자료실은 3층에 있습니다.", "시설"),
    ("매주 토요일 오전에는 어린이 독서 교실 프로그램을 운영합니다.", "프로그램"),
    ("주차장은 지하 1층에 있으며 2시간까지 무료입니다.", "시설"),
]
print("문서 수 →", len(docs))

문서 수 → 6


In [7]:
import numpy as np   # 벡터(숫자 배열) 연산에 사용

# ✅ 포인트: 각 문서를 model.encode로 임베딩 → INSERT. np.asarray(float32)로 vector에 바인딩합니다.
for content, category in docs:
    emb = model.encode(content)                              # 문서 → 1024차원 벡터
    conn.execute(
        # %s는 SQL 플레이스홀더 — 값을 직접 문자열로 이어붙이지 않고 안전하게 채워 넣는 방식입니다.
        "INSERT INTO documents (content, embedding, category) VALUES (%s, %s, %s)",
        (content, np.asarray(emb, dtype=np.float32), category)  # 📌 register_vector 덕에 numpy→vector 바인딩
    )
conn.commit()

cnt = conn.execute("SELECT count(*) FROM documents").fetchone()[0]
print("적재된 문서 수 →", cnt)   # 기대: 6

적재된 문서 수 → 6


### 질의 A — '대출'이라는 단어가 질문에 없는데도?

질문 `"책은 며칠 동안 빌릴 수 있나요?"` 에는 **'대출'** 이라는 단어가 하나도 없습니다. 키워드 검색이라면 못 찾겠죠. 하지만 시맨틱 검색은 **의미**로 찾습니다. 확정 top-1은 **id 3(대출) 문서**입니다.

In [8]:
# ✅ 포인트: 질문도 임베딩해 <=> 코사인 검색 — '거리'가 작을수록 가까움, 1-거리 = 코사인 유사도. 
def semantic_search(query, k=3):  #꼭 k를 인자로 놓아야 하는 건 아니다 - 이 자리에 유사도 최소치 0.6 같은 걸 인자로 설정할 수도 있다
    qe = np.asarray(model.encode(query), dtype=np.float32)   # 질문 임베딩
    rows = conn.execute(
        # <=> 는 pgvector의 코사인 거리 연산자입니다. 값이 작을수록 두 벡터가 더 비슷합니다.
        # 1 - (embedding <=> %s) 로 계산하면 값이 클수록(1에 가까울수록) 더 비슷한 "유사도"가 됩니다.
        "SELECT content, category, 1 - (embedding <=> %s) AS cos_sim "
        "FROM documents ORDER BY embedding <=> %s LIMIT %s",
        (qe, qe, k)   # 같은 질문 벡터(qe)를 SELECT용·ORDER BY용으로 각각 넘겨줍니다.
    ).fetchall()
    return rows

# 질의 A: 질문에 '대출'이 없는데도 '대출' 문서를 1등으로 찾습니다(순위 확정·유사도는 근사).
print("질의 A: 책은 며칠 동안 빌릴 수 있나요?")
for content, category, sim in semantic_search("책은 며칠 동안 빌릴 수 있나요?"):
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")

질의 A: 책은 며칠 동안 빌릴 수 있나요?
  [대출] 유사도 약 0.70  한 번에 최대 5권까지 2주 동안 대출할 수 있습니다.
  [이용안내] 유사도 약 0.55  도서관 운영 시간은 평일 오전 9시부터 오후 10시까지입니다.
  [프로그램] 유사도 약 0.50  매주 토요일 오전에는 어린이 독서 교실 프로그램을 운영합니다.


### 질의 B·C — 시맨틱 검색 감각 익히기

- **질의 B** `"도서관은 몇 시에 문을 닫나요?"` → 확정 top-1 = **id 1(이용안내·운영시간)**. '문을 닫나요' ↔ '운영 시간'이 의미로 매칭됩니다.
- **질의 C** `"노트북 쓸 수 있는 곳이 어디예요?"` → 확정 top-1 = **id 4(시설·디지털 자료실)**.

In [9]:
# 질의 B: '문을 닫나요' ↔ '운영 시간' — 글자는 안 겹쳐도 의미가 매칭됩니다.
print("질의 B: 도서관은 몇 시에 문을 닫나요?")
for content, category, sim in semantic_search("도서관은 몇 시에 문을 닫나요?"):
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")

질의 B: 도서관은 몇 시에 문을 닫나요?
  [이용안내] 유사도 약 0.67  도서관 운영 시간은 평일 오전 9시부터 오후 10시까지입니다.
  [프로그램] 유사도 약 0.50  매주 토요일 오전에는 어린이 독서 교실 프로그램을 운영합니다.
  [시설] 유사도 약 0.45  노트북을 사용할 수 있는 디지털 자료실은 3층에 있습니다.


In [10]:
# 질의 C: '노트북 쓸 수 있는 곳' → 시설(디지털 자료실)을 의미로 찾습니다.
print("질의 C: 노트북 쓸 수 있는 곳이 어디예요?")
for content, category, sim in semantic_search("노트북 쓸 수 있는 곳이 어디예요?"):
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")

질의 C: 노트북 쓸 수 있는 곳이 어디예요?
  [시설] 유사도 약 0.70  노트북을 사용할 수 있는 디지털 자료실은 3층에 있습니다.
  [이용안내] 유사도 약 0.46  도서관 운영 시간은 평일 오전 9시부터 오후 10시까지입니다.
  [프로그램] 유사도 약 0.42  매주 토요일 오전에는 어린이 독서 교실 프로그램을 운영합니다.


> **✅ 포인트**: 이게 **시맨틱 검색** — 키워드가 아니라 **의미**로 찾습니다. 문서를 미리 임베딩해 저장하고, 질문도 임베딩해 가장 가까운 것을 찾는 이 과정이 바로 **RAG의 심장**입니다.
>
> **흔한 실수**: `<=>`는 '유사도'가 아니라 **'거리'** 라 **작을수록 가까움**입니다. 유사도로 보려면 `1 - (embedding <=> %s)` 로 바꿔야 하고, 정렬(`ORDER BY`)은 거리 `<=>` 오름차순 그대로 두면 됩니다.

## 메타데이터 필터 + 벡터 결합 & HNSW 인덱스 (⭐6축 #5 실증)

드디어 앞에서 말한 **⭐6축 #5(메타 필터 + 벡터 결합)** 를 코드로 증명합니다. `WHERE category='시설'`로 **먼저 좁힌 뒤** 벡터로 순위를 매기는 걸 **하나의 SQL로** 합니다.

**질의 D**: `WHERE category='시설'` + `"주차는 어디에 하나요?"` → 시설 2문서(디지털자료실·주차장) 중 확정 top-1 = **id 6(주차장)**.

In [11]:
# ✅ 포인트(⭐6축 #5): WHERE(메타 필터) + 벡터(<=>)를 '하나의 SQL'로 결합합니다.
# 시설 카테고리 2문서(디지털자료실·주차장) 중, '주차' 의미에 가장 가까운 것을 골라냅니다.
qe = np.asarray(model.encode("주차는 어디에 하나요?"), dtype=np.float32)
rows = conn.execute(
    "SELECT content, category, 1 - (embedding <=> %s) AS cos_sim "
    "FROM documents WHERE category = %s "        # ← ① 먼저 카테고리로 좁힌 뒤
    "ORDER BY embedding <=> %s LIMIT 3",          # ← ② 벡터로 순위
    (qe, "시설", qe)
).fetchall()
print("질의 D: WHERE category='시설' + '주차는 어디에 하나요?'")
for content, category, sim in rows:
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")

질의 D: WHERE category='시설' + '주차는 어디에 하나요?'
  [시설] 유사도 약 0.63  주차장은 지하 1층에 있으며 2시간까지 무료입니다.
  [시설] 유사도 약 0.44  노트북을 사용할 수 있는 디지털 자료실은 3층에 있습니다.


> **✅ 포인트(⭐)**: 방금 '정형 필터(`WHERE`)'와 '벡터 유사도(`<=>`)'를 **한 SQL·한 테이블**에서 함께 썼습니다. 이게 앞서 본 6가지 이유 중 **#1(한 곳에 산다)·#5(필터+벡터 결합)** 의 실물 증거예요. 메타데이터와 임베딩이 같은 곳에 사니까 가능한 거죠.

### HNSW 벡터 인덱스 생성 & EXPLAIN — 정직하게 보기

대규모 벡터에서 빠르게 검색하려면 **벡터 인덱스**가 필요합니다. HNSW 인덱스를 만들어 봅니다. 다만 **지금은 6행짜리 작은 표**라, `EXPLAIN`을 보면 planner가 인덱스 대신 **Seq Scan**(전량 비교)을 고릅니다 — JOIN과 인덱스에서 배운 '작은 표는 전부 훑는 게 더 빠르다'는 교훈 그대로입니다. **HNSW의 효과는 문서가 수십만~수백만일 때** 드러납니다.

In [12]:
# ✅ 포인트: HNSW 벡터 인덱스 생성 — 코사인용 opclass는 vector_cosine_ops.
# 📌 빌드는 순식간(데이터 6행·예시) — 실제 이득은 문서가 수십만~수백만일 때.
# 인덱스가 없으면 검색할 때마다 모든 행을 하나씩 비교해야 합니다(느림). HNSW 인덱스를 만들어두면
# PostgreSQL이 "가장 비슷할 만한 후보"만 빠르게 골라내 비교 대상을 크게 줄여줍니다.
conn.execute("CREATE INDEX ON documents USING hnsw (embedding vector_cosine_ops)")
conn.commit()
print("HNSW 인덱스 생성 완료")

HNSW 인덱스 생성 완료


In [13]:
import re   # 문자열에서 패턴을 찾아 바꿔주는 정규표현식 라이브러리 (아래에서 긴 벡터 숫자를 축약하는 데만 사용)

# 📌 정직한 관찰: 6행짜리 작은 표에선 planner가 인덱스 대신 'Seq Scan'(전량 비교)을 고릅니다.
#    JOIN과 인덱스 교훈 그대로 — 작은 표는 전부 훑는 게 더 빠릅니다. HNSW 효과는 대규모에서!
qe = np.asarray(model.encode("책은 며칠 동안 빌릴 수 있나요?"), dtype=np.float32)
# EXPLAIN은 실제로 쿼리를 실행하지 않고, PostgreSQL이 "어떤 방식으로 실행할 계획인지"만 보여줍니다.
# (인덱스를 실제로 쓸지, 아니면 전체를 다 훑을지 등 실행 전략을 미리 확인할 때 사용)
plan = conn.execute(
    "EXPLAIN SELECT content FROM documents ORDER BY embedding <=> %s LIMIT 3",
    (qe,)
).fetchall()

plan_text = "\n".join(line[0] for line in plan)   # 여러 줄로 나온 결과를 하나의 문자열로 합침
# 📌 Sort Key에 질의 벡터(1024개 숫자)가 통째로 찍혀 길어서, '화면 표시용'으로만 축약합니다(계획 자체는 그대로).
# re.sub(패턴, 바꿀문자열, 대상)로, 실행계획 텍스트 안의 긴 벡터 숫자 나열만 짧은 안내문으로 바꿔치기합니다.
plan_text = re.sub(r"'\[[-0-9.,e ]+\]'::vector", "'[...질의벡터 1024차원...]'::vector", plan_text)
print(plan_text)

# ✅ 포인트: 계획에 'Seq Scan on documents'가 보이면 → 인덱스 대신 전량 비교(소형 표라서).
print("\nSeq Scan 사용? →", "Seq Scan" in plan_text)

Limit  (cost=1.15..1.16 rows=3 width=40)
  ->  Sort  (cost=1.15..1.17 rows=6 width=40)
        Sort Key: ((embedding <=> '[...질의벡터 1024차원...]'::vector))
        ->  Seq Scan on documents  (cost=0.00..1.07 rows=6 width=40)

Seq Scan 사용? → True


> **✅ 포인트**: 인덱스는 **만들어 두고**, 효과는 **문서가 많아질 때** 나타납니다(JOIN과 인덱스에서 다룬 대용량 표에서 인덱스가 빛났던 것처럼). 소형 표에서 'HNSW라 빨라졌다'고 말하면 과장이에요 — 정직하게 Seq Scan을 확인했습니다.
>
> 📌 **심화(전공자)**: **HNSW**(계층 그래프 — recall↑·빌드 느림·메모리↑) vs **IVFFlat**(클러스터 후보 — 빌드 빠름·메모리↓·recall/성능↓). '정확도(recall)와 빌드·메모리의 트레이드오프'입니다.

## 오늘 정리 & 다음 단계

- **임베딩** = 텍스트를 '의미의 좌표(1024차원 벡터)'로. `CREATE EXTENSION vector` 한 줄이면 PostgreSQL이 벡터 DB가 됩니다.
- **시맨틱 검색** = 질문을 임베딩해 `<=>` 코사인으로 의미가 가까운 문서를 찾기(키워드가 안 겹쳐도!). **순위는 확정·유사도 수치는 근사**.
- ⭐**PostgreSQL이 임베딩 DB로 좋은 이유 6가지** — 특히 **#1(한 곳에 산다)·#5(필터+벡터 결합)**. `WHERE` + `<=>`를 **한 SQL로** 결합하는 걸 직접 봤습니다.
- **HNSW 인덱스**는 만들어 두되, 효과는 대규모에서(소형 표는 Seq Scan — JOIN과 인덱스 교훈).

### 이제 개인실습을 진행하세요
시연에서 **도서관 문서**로 했던 것을, 이번엔 여러분이 **한빛은행 FAQ/약관 8문서**로 직접 재현합니다 — 임베딩 적재부터 ⭐필터+벡터 결합까지.

**➡️ `pgvector_3_한빛은행_시맨틱_검색_개인실습.ipynb`**

> 이후 또 다른 AI 전용 데이터 — **관계(그래프)** 를 neo4j로 간단히 맛봅니다(부트캠프 후반 온톨로지·Neo4j·GraphRAG 모듈에서 본격적으로 다시 다룹니다). "임베딩은 벡터로(pgvector), 관계는 그래프로(neo4j)".

## 종합 연습 — 상품 설명 시맨틱 검색

쇼핑몰 상품 설명 5개를 임베딩해 넣고, **단어가 아니라 의미로** 상품을 찾아 봅니다. 위에서 쓰던 연결(`conn`)·모델(`model`)을 그대로 씁니다 — 새 표 `products_vec`만 만들고 기존 `documents`는 건드리지 않아요.

정답은 맨 아래 **정답 모음**에 한꺼번에 있어요.

In [14]:
# ✅ 제공 셀 — 그대로 실행: products_vec 표 생성 + 상품 설명 5건 임베딩 적재 (멱등)
conn.execute("DROP TABLE IF EXISTS products_vec")
conn.execute("""
CREATE TABLE products_vec (
    id        BIGSERIAL PRIMARY KEY,
    name      TEXT,                 -- 상품명
    content   TEXT,                 -- 상품 설명(임베딩 대상)
    category  TEXT,                 -- 메타 필터용
    embedding vector(1024)          -- BGE-M3 1024차원
)""")
# 이번 미니 실습에서 사용할 5개 상품 (상품명, 설명, 카테고리) 목록입니다.
items = [
    ("무선 마우스",       "선 없이 블루투스로 연결해 쓰는 컴퓨터 입력 장치", "전자기기"),
    ("기계식 키보드",     "또각또각 눌리는 스위치로 타자를 치는 입력 장치", "전자기기"),
    ("27인치 모니터",     "넓은 화면으로 문서와 영상을 보는 디스플레이",     "전자기기"),
    ("스테인리스 텀블러", "뜨거운 음료를 오래 따뜻하게 보관하는 보온 컵",   "생활용품"),
    ("노트북 백팩",       "노트북을 넣어 등에 메고 다니는 수납 가방",       "생활용품"),
]
# 각 상품의 설명(content)을 임베딩해 products_vec 테이블에 저장합니다. (documents 테이블에 넣던 것과 같은 패턴)
for name, content, catg in items:
    emb = np.asarray(model.encode(content), dtype=np.float32)
    conn.execute("INSERT INTO products_vec (name, content, category, embedding) VALUES (%s, %s, %s, %s)",
                 (name, content, catg, emb))
conn.commit()
print("적재 완료 → products_vec 5행")

적재 완료 → products_vec 5행


In [15]:
# 🧪 Q1 — "선 없이 쓸 수 있는 입력 장치 있나요?" 로 Top-2를 검색하세요 (상품명·유사도)
# 💡 힌트: 질문 임베딩 → SELECT name, 1 - (embedding <=> %s) AS cos_sim ... ORDER BY embedding <=> %s LIMIT 2
# 💡 관찰 포인트: 질문에 '무선'이라는 단어가 없는데도 무선 마우스를 찾아내는가?
# 👇 아래 빈 셀에 직접 작성해 보세요

In [16]:
def semantic_search(query, k=2):  #꼭 k를 인자로 놓아야 하는 건 아니다 - 이 자리에 유사도 최소치 0.6 같은 걸 인자로 설정할 수도 있다
    qe = np.asarray(model.encode(query), dtype=np.float32)   # 질문 임베딩
    rows = conn.execute(
        # <=> 는 pgvector의 코사인 거리 연산자입니다. 값이 작을수록 두 벡터가 더 비슷합니다.
        # 1 - (embedding <=> %s) 로 계산하면 값이 클수록(1에 가까울수록) 더 비슷한 "유사도"가 됩니다.
        "SELECT name, 1 - (embedding <=> %s) AS cos_sim "
        "FROM products_vec ORDER BY embedding <=> %s LIMIT %s",
        (qe, qe, k)   # 같은 질문 벡터(qe)를 SELECT용·ORDER BY용으로 각각 넘겨줍니다.
    ).fetchall()
    return rows

# 질의 A: 질문에 '대출'이 없는데도 '대출' 문서를 1등으로 찾습니다(순위 확정·유사도는 근사).
print("선 없이 쓸 수 있는 입력 장치 있나요?")
for content, sim in semantic_search("선 없이 쓸 수 있는 입력 장치 있나요?"):
    print(f"유사도 약 {sim:.2f}  {content}")

선 없이 쓸 수 있는 입력 장치 있나요?
유사도 약 0.84  무선 마우스
유사도 약 0.58  기계식 키보드


In [17]:
# 🧪 Q2 — 생활용품 중에서 "따뜻한 커피를 담아 다닐 것"에 가장 가까운 상품 1개를 찾으세요
# 💡 힌트: WHERE category = %s 로 먼저 좁히고 → 벡터(<=>)로 순위 (메타 필터 + 벡터 결합)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [21]:
qe = np.asarray(model.encode("따뜻한 커피를 담아 다닐 것"), dtype=np.float32)
rows = conn.execute(
    "SELECT content, category, 1 - (embedding <=> %s) AS cos_sim "
    "FROM products_vec WHERE category = %s "        # ← ① 먼저 카테고리로 좁힌 뒤
    "ORDER BY embedding <=> %s LIMIT 1",          # ← ② 벡터로 순위
    (qe, "생활용품", qe)
).fetchall()
print("WHERE category='생활용품' + '따뜻한 커피를 담아 다닐 것'")
print(rows)

WHERE category='생활용품' + '따뜻한 커피를 담아 다닐 것'
[('뜨거운 음료를 오래 따뜻하게 보관하는 보온 컵', '생활용품', 0.6377636194229126)]


### 정답 모음 — 직접 푼 뒤에 확인하세요

(cos_sim 수치는 실행 환경에서 확인하세요 — 순위가 채점 기준입니다)

In [18]:
# ── Q1 정답 ──
# 아래는 실행 코드가 아니라 "정답 예시"를 주석으로 남겨둔 것입니다 — 필요하면 줄 앞의 #을 지우고 실행해 보세요.
#
# qe = np.asarray(model.encode("선 없이 쓸 수 있는 입력 장치 있나요?"), dtype=np.float32)
# #  → 질문 문장을 다른 문서들과 똑같은 방식으로 1024차원 벡터로 바꿉니다.
# rows = conn.execute(
#     "SELECT name, 1 - (embedding <=> %s) AS cos_sim "
#     "FROM products_vec ORDER BY embedding <=> %s LIMIT 2",
#     (qe, qe)
# ).fetchall()
# #  → <=>(코사인 거리)로 정렬해 가장 비슷한 상품 2개(Top-2)를 가져옵니다. 1 - 거리 = 유사도(클수록 비슷함).
# #     필터(WHERE) 없이 products_vec 전체를 대상으로 검색하는, 가장 기본적인 시맨틱 검색 형태입니다.
# for r in rows:
#     print(r)
# → 1위 '무선 마우스' — 질문에 '무선'이 없어도(선 없이) 의미로 찾습니다. 2위는 기계식 키보드(같은 입력 장치)

In [19]:
# ── Q2 정답 ──
# 아래도 실행 코드가 아니라 "정답 예시" 주석입니다 — 필요하면 줄 앞의 #을 지우고 실행해 보세요.
#
# qe = np.asarray(model.encode("따뜻한 커피를 담아 다닐 것"), dtype=np.float32)
# #  → 질문 문장을 벡터로 변환합니다. (Q1과 같은 방식)
# rows = conn.execute(
#     "SELECT name, category, 1 - (embedding <=> %s) AS cos_sim "
#     "FROM products_vec WHERE category = %s "
#     "ORDER BY embedding <=> %s LIMIT 1",
#     (qe, "생활용품", qe)
# ).fetchall()
# #  → 이번엔 Q1과 달리 WHERE category = %s 로 "생활용품"만 먼저 골라낸 뒤, 그 안에서만 벡터로 순위를 매깁니다.
# #     (실습 힌트에서 말한 "메타 필터 + 벡터 결합" 패턴 — pgvector_1·pgvector_3에서도 반복해서 나온 패턴입니다.)
# print(rows)
# → '스테인리스 텀블러' — ① 카테고리로 좁히고 ② 벡터로 순위 (⭐6축 #5 그대로)